# Bronze Source Exploration

## Clinical Trial Intelligence Platform

### Objective

This notebook performs exploratory validation of the clinical source data available in the Amazon S3 landing zone before production Bronze ingestion.

The notebook is used to:

- verify connectivity to the S3 landing zone,
- inspect representative source records,
- understand the incoming schema,
- profile source-level data characteristics,
- and validate assumptions required by the Bronze ingestion pipeline.

> **Note:** Direct `spark.read` operations in this notebook are used only for source exploration and connectivity validation. Production Bronze ingestion is implemented separately using incremental Auto Loader-based pipelines for operational clinical feeds and batch-oriented processing for master/reference datasets.

## 1. Source Configuration

The following Amazon S3 path contains the EDC subject source files used for source-level exploration.

In [0]:
source_path = "s3://clinical-trial-intelligence-platform-sk/Landing/EDC/subjects/"

## 2. Source Connectivity Validation

Perform a direct batch read to verify that the Databricks environment can access the EDC subject landing path and inspect representative source records.

In [0]:
subjects_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

display(subjects_df.limit(10))

## 3. Source Schema Inspection

Inspect the inferred source schema to understand the fields and datatypes present in the incoming EDC subject files.

In [0]:
subjects_df.printSchema()

## 4. Source Record Profiling

Perform lightweight source profiling to understand the volume and basic structure of the landed subject data.

In [0]:
from pyspark.sql import functions as F

subjects_df.select(
    F.count("*").alias("record_count"),
    F.countDistinct("subject_id").alias("distinct_subjects"),
    F.countDistinct("study_id").alias("distinct_studies"),
    F.countDistinct("site_id").alias("distinct_sites")
).display()

## 5. Exploration Summary

The source exploration confirms that:

- the EDC subject landing path is accessible from Databricks,
- representative source records can be read successfully,
- the incoming schema and key fields can be inspected before pipeline development,
- and the source structure is suitable for downstream Bronze ingestion.

Production ingestion is handled separately by the Bronze pipeline, where operational clinical feeds are processed incrementally using Auto Loader and source-level lineage metadata is captured.